In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_115_NSIT_Dwarka_Delhi_CPCB_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,182.82,260.78,11.08,33.10,25.92,30.31,6.79,1.59,25.41,...,4.47,NaN,78.64,0.68,243.46,NaN,NaN,42.67,NaN,0.22
1,2024-01-02,197.14,298.04,12.58,32.11,26.79,30.42,11.08,1.59,32.89,...,4.13,NaN,77.01,0.67,239.70,NaN,NaN,52.17,NaN,0.03
2,2024-01-03,193.88,269.78,12.41,34.24,27.50,30.01,8.47,1.69,27.18,...,4.10,NaN,89.02,0.51,154.82,NaN,NaN,40.70,NaN,0.06
3,2024-01-04,205.37,285.42,15.46,32.38,29.43,33.33,12.82,1.06,20.17,...,4.12,NaN,97.13,0.56,291.27,NaN,NaN,14.59,NaN,0.15
4,2024-01-05,141.38,217.39,10.62,30.61,24.19,48.59,10.43,1.75,20.17,...,2.76,NaN,96.53,0.52,205.42,NaN,NaN,15.80,NaN,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,88.87,207.80,24.01,46.84,44.34,36.54,12.82,1.16,14.42,...,1.68,NaN,90.51,0.71,124.42,NaN,NaN,6.46,NaN,-0.01
362,2024-12-28,55.08,117.37,23.86,45.07,43.35,36.35,12.65,0.87,14.12,...,1.67,NaN,94.27,0.36,194.45,NaN,NaN,8.68,NaN,0.02
363,2024-12-29,53.17,118.80,23.31,40.12,40.29,32.94,12.39,0.93,14.47,...,1.66,NaN,92.45,0.78,312.25,NaN,NaN,10.41,NaN,0.12
364,2024-12-30,49.95,123.37,23.32,41.16,40.85,33.07,12.67,0.99,15.26,...,1.66,NaN,86.36,0.57,278.34,NaN,NaN,12.25,NaN,0.15


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)', 'BP (mmHg)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Toluene (µg/m³)        0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
RH (%)                 0
WS (m/s)               0
WD (deg)               0
SR (W/mt2)             0
VWS (m/s)              0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 19)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         182.82        260.78       11.08        33.10   
1  2024-01-02         197.14        298.04       12.58        32.11   
2  2024-01-03         193.88        269.78       12.41        34.24   
3  2024-01-04         205.37        285.42       15.46        32.38   
4  2024-01-05         141.38        217.39       10.62        30.61   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      25.92        30.31         6.79        1.59          25.41   
1      26.79        30.42        11.08        1.59          32.89   
2      27.50        30.01         8.47        1.69          27.18   
3      29.43        33.33        12.82        1.06          20.17   
4      24.19        48.59        10.43        1.75          20.17   

   Benzene (µg/m³)  Toluene (µg/m³)  Eth-Benzene (µg/m³)  MP-Xylene (µg/m³)  \
0             7.28             9.37                 4.92

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),RH (%),WS (m/s),WD (deg),SR (W/mt2),VWS (m/s)
0,2024-01-01,1.631203,0.424686,-0.642255,0.225876,-0.181411,-0.984283,-1.790654,2.137208,-0.063783,2.369103,1.135551,2.388434,0.805357,0.890357,0.079013,0.642843,-1.304431,1.618601
1,2024-01-02,1.897247,0.764306,-0.344665,0.142406,-0.098071,-0.971541,-0.597014,2.137208,0.502772,2.090775,0.941707,2.138842,0.606724,0.815518,0.047511,0.578450,-1.094790,-0.436118
2,2024-01-03,1.836681,0.506720,-0.378392,0.321993,-0.030057,-1.019033,-1.323215,2.502833,0.070281,2.071798,0.907092,2.052445,0.589197,1.366940,-0.456521,-0.875187,-1.347904,-0.111689
3,2024-01-04,2.050148,0.649276,0.226708,0.165171,0.154825,-0.634466,-0.112880,0.199396,-0.460675,2.071798,0.931322,2.090843,0.600881,1.739300,-0.299011,1.461628,-1.924086,0.861599
4,2024-01-05,0.861310,0.029191,-0.733516,0.015937,-0.347135,1.133151,-0.777869,2.722208,-0.460675,1.116629,0.280561,1.495663,-0.193654,1.711752,-0.425019,-0.008622,-1.897384,1.402315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,-0.114247,-0.058221,1.922970,1.384340,1.583110,-0.262641,-0.112880,0.565021,-0.896195,-0.559662,-0.432507,-0.529871,-0.824608,1.435352,0.173519,-1.395811,-2.103494,-0.868691
362,2024-12-28,-0.742015,-0.882481,1.893211,1.235105,1.488274,-0.284649,-0.160181,-0.495292,-0.918918,-0.584965,-0.456738,-0.558670,-0.830450,1.607987,-0.929051,-0.196492,-2.054504,-0.544261
363,2024-12-29,-0.777500,-0.869446,1.784095,0.817755,1.195145,-0.679641,-0.232522,-0.275917,-0.892408,-0.584965,-0.456738,-0.549071,-0.836292,1.524424,0.394033,1.820927,-2.016328,0.537170
364,2024-12-30,-0.837323,-0.827791,1.786079,0.905441,1.248790,-0.664583,-0.154616,-0.056542,-0.832571,-0.584965,-0.456738,-0.549071,-0.836292,1.244810,-0.267509,1.240191,-1.975723,0.861599


In [10]:
df.to_excel('NSIT2024.xlsx', index=False)